<a href="https://colab.research.google.com/github/Rubal-code/DEEP-LEARNING/blob/main/cliff_walking_reinforcement-learning_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import gymnasium as gym
import numpy as np
import cv2
import imageio
from IPython.display import Image
from google.colab.patches import cv2_imshow

In [27]:
import sys
!{sys.executable} -m pip install gymnasium --force-reinstall
# IMPORTANT: After running this cell, please restart the Colab runtime (Runtime -> Restart runtime).

  Using cached gymnasium-1.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached numpy-2.5.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached farama_notifications-0.0.6-py3-none-any.whl.metadata (729 bytes)
Using cached gymnasium-1.3.0-py3-none-any.whl (953 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached farama_notifications-0.0.6-py3-none-any.whl (2.9 kB)
Using cached numpy-2.5.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
Using cached typing_extensions-4.16.0-py3-none-any.whl (45 kB)
  Attempting uninstall: farama-notifications
    Found existing installation: Farama-Notifications 0.0.6
    Uninstalling Farama-Notifications-0.0.6:
      Successfully uninstalled Farama-Notifications-0.0.6
  Attempting uninstall: typing-extensions
    Found existing ins

In [28]:
cliff_env=gym.make("CliffWalking-v1", render_mode="ansi")

In [29]:
# Creates cliff walking grid
def initialize_frame():
    width, height = 600, 200
    img = np.ones(shape=(height, width, 3)) * 255.0
    margin_horizontal = 6
    margin_vertical = 2

    # Vertical Lines
    for i in range(13):
        img = cv2.line(img, (49 * i + margin_horizontal, margin_vertical),
                       (49 * i + margin_horizontal, 200 - margin_vertical), color=(0, 0, 0), thickness=1)

    # Horizontal Lines
    for i in range(5):
        img = cv2.line(img, (margin_horizontal, 49 * i + margin_vertical),
                       (600 - margin_horizontal, 49 * i + margin_vertical), color=(0, 0, 0), thickness=1)

    # Cliff Box
    img = cv2.rectangle(img, (49 * 1 + margin_horizontal + 2, 49 * 3 + margin_vertical + 2),
                        (49 * 11 + margin_horizontal - 2, 49 * 4 + margin_vertical - 2), color=(255, 0, 255),
                        thickness=-1)
    img = cv2.putText(img, text="Cliff", org=(49 * 5 + margin_horizontal, 49 * 4 + margin_vertical - 10),
                      fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=1, color=(255, 255, 255), thickness=2)

    # Goal
    frame = cv2.putText(img, text="G", org=(49 * 11 + margin_horizontal + 10, 49 * 4 + margin_vertical - 10),
                        fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=1, color=(0, 0, 0), thickness=2)
    # Start
    # frame = cv2.putText(img, text="S", org=(49 * 0 + margin_horizontal + 10, 49 * 4 + margin_vertical - 10),
    #                     fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=1, color=(0, 0, 0), thickness=2)
    return frame


In [30]:
# puts the agent at a state
def put_agent(img, state):
    margin_horizontal = 6
    margin_vertical = 2
    row, column = np.unravel_index(indices=state, shape=(4, 12))
    cv2.putText(img, text="A", org=(49 * column + margin_horizontal + 10, 49 * (row + 1) + margin_vertical - 10),
                fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=1, color=(0, 0, 0), thickness=2)
    return img

In [31]:
done=False
frame = initialize_frame()
observation, info = cliff_env.reset()
state = observation

In [ ]:
frames = [] # List to store frames for GIF

while not done:
  frame2 = put_agent(frame.copy(), state)
  frames.append(frame2) # Append frame to list
  # cv2_imshow(frame2) # Removed direct display
  # cv2.waitKey(250)    # Removed waitKey
  action = int(np.random.randint(low=0, high=4, size=1)[0])
  state, reward, terminated, truncated, info = cliff_env.step(action)
  done = terminated or truncated

cv2.destroyAllWindows() # Not strictly necessary for cv2_imshow, but good practice
cliff_env.close()

# Save frames as GIF and display
imageio.mimsave('cliff_walking.gif', frames, fps=4) # Adjust fps as needed
Image(open('cliff_walking.gif', 'rb').read())

### Explanation of the `while not done:` loop and Visualization:

This code block simulates steps in the `CliffWalking-v1` environment until a terminal state is reached (`done` becomes `True`). Instead of displaying each frame individually, it collects them to create an animated GIF.

*   **`frames = []`**: Initializes an empty list to store each visual frame of the simulation.
*   **`while not done:`**: This loop continues as long as the agent has not reached a terminal state (either falling off the cliff or reaching the goal) or the episode has not been truncated (e.g., due to a time limit).
*   **`frame2 = put_agent(frame.copy(), state)`**: Creates a copy of the base environment grid (`frame`) and draws the agent's current position onto it, returning the updated `frame2`.
*   **`frames.append(frame2)`**: Each generated `frame2` is added to the `frames` list. These collected frames will later be used to create the animation.
*   **`action = int(np.random.randint(low=0, high=4, size=1)[0])`**: This randomly selects an action for the agent. In `CliffWalking-v1`, there are 4 possible actions (0, 1, 2, 3), typically corresponding to up, down, left, and right. `np.random.randint` generates a random integer within a NumPy array, and `[0]` extracts the scalar value, which is then converted to a standard Python integer.
*   **`state, reward, terminated, truncated, info = cliff_env.step(action)`**: This is the core interaction with the environment. The `step()` method takes an `action` and returns five values:
    *   `state`: The new observation of the agent after taking the action.
    *   `reward`: The reward received for taking that action (e.g., -1 for a normal step, -100 for falling off the cliff, +100 for reaching the goal).
    *   `terminated`: A boolean indicating if the episode has ended because the agent reached a terminal state (e.g., fell off a cliff or reached the goal).
    *   `truncated`: A boolean indicating if the episode has ended because of a time limit or other truncation condition.
    *   `info`: Additional information (often empty or for debugging).
*   **`done = terminated or truncated`**: Combines the `terminated` and `truncated` flags to determine if the episode is considered complete for the `while` loop condition.
*   **`cv2.destroyAllWindows()`**: After the loop finishes, this is included for completeness, although typically not strictly needed when using `cv2_imshow` in Colab.
*   **`cliff_env.close()`**: Closes the environment, releasing any associated resources.
*   **`imageio.mimsave('cliff_walking.gif', frames, fps=4)`**: After the simulation, all collected `frames` are saved as an animated GIF file named `cliff_walking.gif`. The `fps` (frames per second) parameter controls the animation speed.
*   **`Image(open('cliff_walking.gif', 'rb').read())`**: The generated GIF file is then read and displayed directly in the Colab output as a single, animated image.